# Variant B — MeanFlow Decoder (corrected)

Replaces the deterministic CSINet+ decoder with a one-step MeanFlow generative decoder (trained with the MeanFlow Identity and forward-mode JVP). This corrected version restores gradient flow through the STN and uses the proper one-step sampling direction.

This notebook is a standalone, simplified version of `varB_mean_flow_corrected.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
class TrainingConfig:
    train_file:        str   = "train_data.mat"
    val_file:          str   = "val_data.mat"
    test_file:         str   = "test_data.mat"
    checkpoint_dir:    str   = "checkpoints_mean_flow_corrected"
    fast_dev_run:      bool  = False
    run_training:      bool  = False
    epochs:            int   = 500
    batch_size:        int   = 200
    learning_rate:     float = 0.001
    min_lr:            float = 0.0001
    patience:          int   = 20
    weight_decay:      float = 1e-5
    grad_clip:         float = 1.0
    snr_low:           float = -10.0
    snr_high:          float = 10.0
    k_feedback:        int   = 64
    compression_ratio: int   = 16
    # Loss weights — flow trains the MeanFlow identity;
    # mse trains the end-to-end reconstruction quality
    mse_weight:        float = 0.7
    flow_weight:       float = 0.3
    warmup_epochs:     int   = 20   # mse-only until this epoch
    # MeanFlow hyper-parameters
    T_infer:           int   = 1    # inference steps (1 = one-shot, as paper)
    flow_channels:     int   = 32   # hidden channels in velocity network
    t_dim:             int   = 64   # time embedding dimension
    jvp_eps:           float = 1e-3 # finite-difference step for JVP
    r_neq_t_ratio:     float = 0.75 # fraction of training steps where r ≠ t
    save_every:        int   = 10
    run_evaluation:    bool  = False
    checkpoint_path:   str | None = None
    resume_latest:     bool  = False
cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def _read_scalar(d):
    v = d[()]
    return float(v.reshape(-1)[0]) if isinstance(v, np.ndarray) and v.size == 1 else v

def load_dataset_cfg(path):
    with h5py.File(path, "r") as f:
        g = f["cfg"]
        return {k: _read_scalar(g[k]) for k in g if isinstance(g[k], h5py.Dataset)}

def get_train_global_scale(path, chunk=500):
    stats = {"dl": {"sum_sq": 0., "count": 0}, "ul": {"sum_sq": 0., "count": 0}}
    with h5py.File(path, "r") as f:
        for key, sn in [("csi_dl", "dl"), ("csi_ul", "ul")]:
            ds = f[key]; N = ds.shape[3]
            for s in range(0, N, chunk):
                c = ds[:, :, :, s:min(s+chunk, N)]
                r = c["real"].astype(np.float32); im = c["imag"].astype(np.float32)
                stats[sn]["sum_sq"] += float(np.sum(r**2) + np.sum(im**2))
                stats[sn]["count"]  += r.size + im.size
    out = {k: {"std": float(np.sqrt(stats[k]["sum_sq"] / max(stats[k]["count"], 1) + 1e-12))}
           for k in ("dl", "ul")}
    print("Scale:", out); return out

class CSIDatasetManager:
    def __init__(self, tp, vp, tsp, stats):
        self.stats = stats; self.files = {}; self.datasets = {}; self.lengths = {}
        for split, path in [("train", tp), ("val", vp), ("test", tsp)]:
            h = h5py.File(path, "r"); self.files[split] = h
            self.datasets[split] = {"dl": h["csi_dl"], "ul": h["csi_ul"]}
            self.lengths[split]  = int(h["csi_dl"].shape[3]); print(f"{split}: {self.lengths[split]}")
    def close(self): [h.close() for h in self.files.values()]
    def _norm(self, a, k):  return a / (self.stats[k]["std"] + 1e-8)
    def denorm(self, t, k): return t  * (self.stats[k]["std"] + 1e-8)
    def _proc(self, a, k, norm=True):
        r = a["real"].astype(np.float32); im = a["imag"].astype(np.float32)
        if norm: r = self._norm(r, k); im = self._norm(im, k)
        return np.transpose(np.squeeze(np.stack([r, im], 2), 3), (3, 2, 0, 1))
    def get_batch(self, split, idx, sv=None):
        idx = np.sort(np.asarray(idx, dtype=np.int64))
        dl = torch.from_numpy(self._proc(self.datasets[split]["dl"][:,:,:,idx], "dl")).float()
        ul = torch.from_numpy(self._proc(self.datasets[split]["ul"][:,:,:,idx], "ul")).float()
        if sv is None: sv = np.random.uniform(cfg.snr_low, cfg.snr_high, (len(idx), 1)).astype(np.float32)
        else:          sv = np.asarray(sv, dtype=np.float32).reshape(len(idx), 1)
        return dl, ul, torch.from_numpy(sv).float()
    def iterate(self, split, bs, shuffle=False, rng=None, fixed_snr=None):
        N = self.lengths[split]; order = np.arange(N, dtype=np.int64)
        if shuffle: (rng if rng else np.random.default_rng()).shuffle(order)
        for s in range(0, N, bs):
            bi = order[s:s+bs]
            sv = None if fixed_snr is None else np.full((len(bi),1), fixed_snr, dtype=np.float32)
            yield self.get_batch(split, bi, sv)

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    def __init__(self, ch, r=2):
        super().__init__(); h = max(ch//r, 1)
        self.fc1 = nn.Linear(ch+1, h); self.fc2 = nn.Linear(h, ch)
    def forward(self, x, snr):
        p = F.adaptive_avg_pool2d(x, 1).flatten(1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(torch.cat([p, snr], 1)))))
        return x * s.view(x.size(0), x.size(1), 1, 1)

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Conv2d(2,16,3,stride=(2,1),padding=1); self.b1=nn.BatchNorm2d(16); self.p1=nn.PReLU(); self.a1=AFModule(16)
        self.c2=nn.Conv2d(16,16,3,stride=(2,1),padding=1); self.b2=nn.BatchNorm2d(16); self.p2=nn.PReLU(); self.a2=AFModule(16)
        self.c3=nn.Conv2d(16,2,3,stride=(2,1),padding=1); self.b3=nn.BatchNorm2d(2)
    def forward(self, x, snr):
        x=self.a1(self.p1(self.b1(self.c1(x))),snr); x=self.a2(self.p2(self.b2(self.c2(x))),snr)
        return self.b3(self.c3(x))

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, cr):
        super().__init__(); self.M=(2*32*32)//cr
        self.c1=nn.Conv2d(2,2,7,padding=3); self.b1=nn.BatchNorm2d(2); self.a1=AFModule(2)
        self.c2=nn.Conv2d(2,2,7,padding=3); self.b2=nn.BatchNorm2d(2); self.a2=AFModule(2)
        self.fc=nn.Linear(2*32*32,self.M)
    def forward(self, x, snr):
        x=self.a1(F.leaky_relu(self.b1(self.c1(x)),0.3),snr)
        x=self.a2(F.leaky_relu(self.b2(self.c2(x)),0.3),snr); return self.fc(x.flatten(1))

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc2cplx(e):
    k=e.shape[1]//2; s=torch.complex(e[:,:k],e[:,k:])
    return s/torch.sqrt(torch.mean(s.abs().square(),1,keepdim=True)+1e-8)

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannel(nn.Module):
    def __init__(self, Nt=32, trs=True): super().__init__(); self.Nt=Nt; self.trs=trs
    def _idx(self, ns, k, dev):
        if self.training and self.trs: return torch.randperm(ns, device=dev)[:k]
        return torch.linspace(0,ns-1,k,device=dev).round().long()
    def forward(self, s, snr, h):
        bs,k=s.shape; dev=s.device; idx=self._idx(h.shape[2],k,dev)
        hs=h[:,:,idx,:]; hu=torch.complex(hs[:,0],hs[:,1])
        nstd=torch.sqrt(1./torch.pow(10.,snr/10.)/2.).unsqueeze(-1)
        z=torch.complex(torch.randn(bs,k,self.Nt,device=dev)*nstd, torch.randn(bs,k,self.Nt,device=dev)*nstd)
        y=hu*s.unsqueeze(-1)+z; w=hu/(torch.norm(hu,2,keepdim=True)+1e-8)
        return torch.sum(torch.conj(w)*y,2)

## C2R — Complex → real for the decoder

In [ ]:
class C2R(nn.Module):
    def forward(self, s): return torch.cat([s.real, s.imag], 1)

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

### Variant-specific: `SinusoidalEmbedding`

In [ ]:
class SinusoidalEmbedding(nn.Module):
    """Maps a scalar t ∈ [0,1] → R^d using sinusoidal frequencies."""
    def __init__(self, d: int = 64):
        super().__init__(); self.d = d
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        t = t.view(-1)
        half  = self.d // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args  = t.unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)  # (B, d)

### Variant-specific: `TwoTimeEmbedding`

In [ ]:
class TwoTimeEmbedding(nn.Module):
    """
    FIX (Bug 4): embed both r and t separately then fuse.
    The network u_θ(z, r, t) must be differentiable w.r.t. t for the JVP.
    Fusing r and t via an MLP gives the network enough signal to distinguish
    the two time variables, which is essential for the MeanFlow identity.
    Conditioning on (t, t-r) matches the best ablation in the paper (Tab. 1c).
    """
    def __init__(self, d: int = 64):
        super().__init__()
        self.sin_emb = SinusoidalEmbedding(d)
        # Two embeddings concatenated then fused
        self.mlp = nn.Sequential(
            nn.Linear(d * 2, d * 2),
            nn.SiLU(),
            nn.Linear(d * 2, d),
        )

    def forward(self, r: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # Condition on (t, t-r) as the paper's best variant
        t_emb    = self.sin_emb(t)          # (B, d)
        dt_emb   = self.sin_emb(t - r)      # (B, d)  — interval embedding
        return self.mlp(torch.cat([t_emb, dt_emb], dim=1))  # (B, d)

### Variant-specific: `VelocityNet`

In [ ]:
class VelocityNet(nn.Module):
    """
    Average velocity network u_θ(z_t, r, t, c_hat_2d, snr).

    FIX (Bug 4): now takes BOTH r and t via TwoTimeEmbedding.
    FIX (JVP): must be differentiable w.r.t. t (ensured by not stopping
               gradients on t before this forward call).

    Inputs:
      z_t      : (B, 2, 32, 32)  current state on the flow path
      r        : (B,)            start time
      t        : (B,)            current time
      c_hat_2d : (B, 2, 32, 32)  conditioning — received signal projected spatially
      snr      : (B, 1)          channel SNR

    Output: (B, 2, 32, 32) predicted average velocity u_θ(z_t, r, t)
    """
    def __init__(self, ch: int = 32, t_dim: int = 64):
        super().__init__()
        self.time_emb   = TwoTimeEmbedding(d=t_dim)
        self.t_proj     = nn.Linear(t_dim, ch)
        # Concatenate z_t + c_hat_2d → 4-channel input
        self.enc1 = nn.Conv2d(4, ch, 3, padding=1); self.bn1 = nn.BatchNorm2d(ch); self.af1 = AFModule(ch)
        self.enc2 = nn.Conv2d(ch, ch, 3, padding=1); self.bn2 = nn.BatchNorm2d(ch); self.af2 = AFModule(ch)
        self.enc3 = nn.Conv2d(ch, ch, 3, padding=1); self.bn3 = nn.BatchNorm2d(ch); self.af3 = AFModule(ch)
        # FiLM conditioning: time modulates spatial features
        self.film_scale = nn.Linear(ch, ch)
        self.film_shift = nn.Linear(ch, ch)
        # Decode to velocity field
        self.dec1 = nn.Conv2d(ch, ch//2, 3, padding=1); self.dbn1 = nn.BatchNorm2d(ch//2)
        self.dec2 = nn.Conv2d(ch//2, 2, 3, padding=1)

    def forward(self, z_t, r, t, c_hat_2d, snr):
        B = z_t.size(0)
        # Time embedding — both r and t, producing a fused vector
        rt_emb  = self.time_emb(r, t)                           # (B, t_dim)
        t_feat  = F.silu(self.t_proj(rt_emb))                   # (B, ch)
        scale   = self.film_scale(t_feat).view(B, -1, 1, 1)     # (B, ch, 1, 1)
        shift   = self.film_shift(t_feat).view(B, -1, 1, 1)     # (B, ch, 1, 1)

        inp = torch.cat([z_t, c_hat_2d], dim=1)                 # (B, 4, 32, 32)
        x   = self.af1(F.silu(self.bn1(self.enc1(inp))), snr)
        x   = self.af2(F.silu(self.bn2(self.enc2(x))),   snr)
        x   = self.af3(F.silu(self.bn3(self.enc3(x))),   snr)

        # FiLM modulation: insert time information into spatial features
        x   = x * (1.0 + scale) + shift

        x   = F.silu(self.dbn1(self.dec1(x)))
        return self.dec2(x)                                      # (B, 2, 32, 32)

### Variant-specific: `MeanFlowDecoder`

In [ ]:
class MeanFlowDecoder(nn.Module):
    """
    Corrected MeanFlow decoder.

    Conventions (matching the paper):
      data  x = T (ground-truth CSI in transform domain, at t=0)
      noise ε = x_0 = FC(c_hat) (rough received-signal estimate, at t=1)

      Flow path: z_t = (1-t)*T + t*x_0
      Instantaneous velocity: v_t = x_0 - T  (= ε - x, constant along path)
      Average velocity: u_θ(z_t, r, t) — what the network learns

    MeanFlow identity (paper Eq. 6):
      u_tgt = v_t - (t-r) * (v_t·∂_z u_θ + ∂_t u_θ)
            = v_t - (t-r) * JVP(u_θ, (z_t,r,t), tangent=(v_t,0,1))

    JVP via finite differences (avoids torch.func complications):
      JVP ≈ [u_θ(z_t + eps·v_t, r, t+eps) - u_θ(z_t, r, t)] / eps
      u_tgt is stop-gradiented (per paper: sg(u_tgt))

    Inference (paper Alg. 2, one step):
      T_hat = x_0 - 1·u_θ(x_0, r=0, t=1)
            = FC(c_hat) - u_θ(FC(c_hat), 0, 1)

    FIX Bug 2: T_hat is computed WITH gradients during training so the full
    chain (ATN → encoder → decoder → STN) receives gradient from the MSE loss.
    """

    def __init__(self, input_dim: int, ch: int = 32, t_dim: int = 64):
        super().__init__()
        flat          = 2 * 32 * 32
        self.fc       = nn.Linear(input_dim, flat)   # rough estimate projection
        self.c_proj   = nn.Linear(input_dim, flat)   # conditioning projection
        self.vel      = VelocityNet(ch=ch, t_dim=t_dim)

    # ── helpers ─────────────────────────────────────────────────────────────

    def get_x0_and_cond(self, c_hat, snr):
        """
        Returns:
          x_0      : (B,2,32,32)  rough estimate = FC(c_hat)  [the "noise" at t=1]
          c_hat_2d : (B,2,32,32)  spatial conditioning
        """
        x_0      = self.fc(c_hat).view(-1, 2, 32, 32)
        c_hat_2d = self.c_proj(c_hat).view(-1, 2, 32, 32)
        return x_0, c_hat_2d

    # ── MeanFlow training loss ───────────────────────────────────────────────

    def meanflow_loss(self, c_hat: torch.Tensor, snr: torch.Tensor,
                      T_gt: torch.Tensor) -> torch.Tensor:
        """
        Compute the MeanFlow training loss on a batch.

        Args:
          c_hat : (B, M)      received signal (encoder output after MRC+C2R)
          snr   : (B, 1)      per-sample SNR
          T_gt  : (B,2,32,32) ground-truth CSI in transform domain (stop-gradiented)

        Returns:
          scalar flow loss
        """
        x_0, c_hat_2d = self.get_x0_and_cond(c_hat, snr)

        B = x_0.shape[0]; dev = x_0.device

        # ── Sample (r, t) pairs ────────────────────────────────────────────
        # Following the paper: sample both uniformly then assign larger to t.
        # With r_neq_t_ratio probability, r ≠ t; otherwise r = t (reduces to FM).
        t = torch.rand(B, device=dev)
        r = torch.rand(B, device=dev) * t  # r ~ U(0, t), so r < t always

        # Enforce r == t for a fraction of samples (r=t term → pure flow matching)
        mask_same = torch.rand(B, device=dev) > cfg.r_neq_t_ratio
        r = torch.where(mask_same, t, r)   # r = t for ~25% of samples

        # ── Flow path interpolation ────────────────────────────────────────
        # z_t = (1-t)*T_gt + t*x_0
        # (paper: z_t = (1-t)*data + t*noise, t=0→data, t=1→noise)
        t4d  = t.view(B, 1, 1, 1)
        r4d  = r.view(B, 1, 1, 1)
        T_d  = T_gt.detach()
        x0_d = x_0.detach()

        z_t  = (1.0 - t4d) * T_d + t4d * x0_d          # (B,2,32,32)
        v_t  = x0_d - T_d                                # instantaneous vel (B,2,32,32)

        # ── JVP via finite differences (FIX Bug 1) ─────────────────────────
        # JVP(u_θ, (z_t,r,t), tangent=(v_t,0,1))
        # ≈ [u_θ(z_t + eps*v_t, r, t+eps) - u_θ(z_t, r, t)] / eps
        # The JVP is used only inside the stop-gradiented target, so we
        # compute it under no_grad to avoid unnecessary memory use.
        eps = cfg.jvp_eps
        with torch.no_grad():
            z_pert = z_t + eps * v_t                        # perturb z
            t_pert = (t + eps).clamp(0.0, 1.0)             # perturb t

            u_val  = self.vel(z_t,     r, t,      c_hat_2d.detach(), snr.detach())
            u_pert = self.vel(z_pert,  r, t_pert, c_hat_2d.detach(), snr.detach())

            jvp    = (u_pert - u_val) / eps                 # (B,2,32,32)

            # MeanFlow target: u_tgt = v_t - (t-r)*JVP
            dt      = (t4d - r4d)                           # (B,1,1,1)
            u_tgt   = v_t - dt * jvp                        # (B,2,32,32)
            # u_tgt is already fully detached (stop-gradient)

        # ── Regression loss: u_θ → sg(u_tgt) ─────────────────────────────
        # Now compute u_θ WITH gradients so the loss backprops into the network
        u_theta = self.vel(z_t, r, t, c_hat_2d.detach(), snr.detach())
        return F.mse_loss(u_theta, u_tgt)

    # ── Inference ────────────────────────────────────────────────────────────

    def forward(self, c_hat: torch.Tensor, snr: torch.Tensor,
                T_steps: int | None = None) -> torch.Tensor:
        """
        One-step (or T_steps Euler) MeanFlow inference.

        Paper Alg. 2 (one step):
          T_hat = x_0 - (t-r) * u_θ(x_0, r=0, t=1)
                = FC(c_hat) - 1 * u_θ(FC(c_hat), r=0, t=1)

        FIX Bug 3 (direction): we START from x_0 (the noise/prior at t=1) and
        SUBTRACT the average velocity to move TOWARD T (the data at t=0).

        FIX Bug 2 (gradient): this forward pass is differentiable w.r.t. all
        parameters, so the MSE loss on stn(T_hat, snr) backprops through here.
        """
        T = T_steps if T_steps is not None else cfg.T_infer
        x_0, c_hat_2d = self.get_x0_and_cond(c_hat, snr)
        B = x_0.shape[0]; dev = x_0.device

        z = x_0  # start at t=1 (the rough estimate / "noise")

        # Euler integration from t=1 to r=0, stepping BACKWARD in time
        t_start = 1.0
        t_end   = 0.0
        step_size = (t_start - t_end) / T  # positive step backward

        for i in range(T):
            t_cur = torch.full((B,), t_start - i * step_size, device=dev)
            r_cur = torch.full((B,), t_start - (i+1) * step_size, device=dev)
            r_cur = r_cur.clamp(0.0, 1.0)

            u = self.vel(z, r_cur, t_cur, c_hat_2d, snr)
            # Displacement backward: z_{r} = z_t - (t-r)*u
            z = z - step_size * u

        return z   # T_hat ≈ T (ground-truth CSI in transform domain)

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.t1=nn.ConvTranspose2d(2,16,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b1=nn.BatchNorm2d(16); self.p1=nn.PReLU(); self.a1=AFModule(16)
        self.t2=nn.ConvTranspose2d(16,16,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b2=nn.BatchNorm2d(16); self.p2=nn.PReLU(); self.a2=AFModule(16)
        self.t3=nn.ConvTranspose2d(16,2,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b3=nn.BatchNorm2d(2)
    def forward(self, x, snr):
        x=self.a1(self.p1(self.b1(self.t1(x))),snr); x=self.a2(self.p2(self.b2(self.t2(x))),snr)
        return self.b3(self.t3(x))

## Build the modules

In [ ]:
dataset_cfg = load_dataset_cfg(train_file)
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannel(num_bs_antennas=int(dataset_cfg['num_bs_antennas'])).to(device)
c2r = C2R().to(device)
decoder = MeanFlowDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def nmse_db_from_sums(e, p):
    return 10.0 * math.log10((e / max(p, 1e-12)) + 1e-12)

def run_epoch(split, epoch_index=0, fixed_snr=None):
    is_train = split == "train"
    for m in [atn, encoder, decoder, stn, ch_sim]: m.train(is_train)
    rng  = np.random.default_rng(SEED + epoch_index)
    tl = tm = ts = es = ps = fa = 0.

    for bi, (H_d, H_u, snr) in enumerate(
        dataset.iterate(split, cfg.batch_size, shuffle=is_train,
                        rng=rng if is_train else None, fixed_snr=fixed_snr)
    ):
        if is_train and DBT and bi >= DBT: break
        if not is_train and DBE and bi >= DBE: break
        H_d=H_d.to(device,non_blocking=True); H_u=H_u.to(device,non_blocking=True)
        snr=snr.to(device,non_blocking=True)

        if is_train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            H_hat, loss, ml, nl, fl = forward_and_loss(H_d, H_u, snr, epoch_index)
            if is_train:
                loss.backward()
                if cfg.grad_clip > 0: nn.utils.clip_grad_norm_(all_params, cfg.grad_clip)
                optimizer.step()

        b  = H_d.size(0); tl += float(loss.detach())*b; tm += float(ml)*b
        fa += float(fl)*b; ts += b
        Hdd = dataset.denorm(H_d.detach(), "dl"); Hhd = dataset.denorm(H_hat.detach(), "dl")
        es += float(torch.sum((Hdd-Hhd)**2)); ps += float(torch.sum(Hdd**2))

    return {"loss": tl/max(ts,1), "mse": tm/max(ts,1), "flow_loss": fa/max(ts,1),
            "nmse_db": nmse_db_from_sums(es, ps), "linear_nmse": es/max(ps,1e-12),
            "samples": int(ts)}

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```